# Genetic Algorithm (GA) for the Traveling Salesman Problem (TSP)

## Objective function
The goal is to find the shortest closed tour that visits every city exactly once and returns to the starting city:

**Minimize** `total_distance = distance(city 1, city 2) + ... + distance(last city, first city)`.

This example uses Euclidean distance between sample city coordinates. You can change the city coordinates in the next cell. A chromosome is a permutation of city indices, and the fitness is `1 / total_distance` (higher fitness means a shorter tour).

Run the cells from top to bottom.

## 1. Define the cities

These are example coordinates, not real geographic locations. Each city is represented by an `(x, y)` point.

In [5]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

# Change these coordinates if your teacher provides a different set of cities.
cities = {
    'A': (0, 0),
    'B': (2, 6),
    'C': (5, 5),
    'D': (6, 1),
    'E': (8, 4),
    'F': (1, 3),
    'G': (4, 0),
    'H': (7, 7)
}

city_names = list(cities.keys())
coordinates = np.array([cities[name] for name in city_names], dtype=float)
n_cities = len(city_names)

print('Cities:', city_names)
print('Number of cities:', n_cities)

Cities: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
Number of cities: 8


## 2. Objective and fitness functions

`route_distance(route)` calculates the complete tour length, including the return from the last city to the first. The GA minimizes distance; `fitness(route)` converts it into a score to maximize.

In [6]:
def route_distance(route):
    """Total distance of a closed tour; route contains each city index once."""
    total = 0.0
    for i in range(len(route)):
        current_city = coordinates[route[i]]
        next_city = coordinates[route[(i + 1) % len(route)]]
        total += math.dist(current_city, next_city)
    return total

def fitness(route):
    """Higher fitness corresponds to a shorter route."""
    distance = route_distance(route)
    return 1.0 / distance if distance > 0 else float('inf')

# Example: calculate a tour's distance
example_route = list(range(n_cities))
print('Example tour:', ' -> '.join(city_names[i] for i in example_route + [example_route[0]]))
print(f'Example total distance: {route_distance(example_route):.3f}')

Example tour: A -> B -> C -> D -> E -> F -> G -> H -> A
Example total distance: 46.044


## 3. Enter the GA parameters

Press Enter to use each suggested default. For a small classroom example, a population of 100 and 300 generations are reasonable starting values.

In [4]:
def read_int(prompt, default, minimum):
    raw = input(f'{prompt} [default {default}]: ').strip()
    value = default if raw == '' else int(raw)
    if value < minimum:
        raise ValueError(f'{prompt} must be at least {minimum}.')
    return value

def read_probability(prompt, default):
    raw = input(f'{prompt} [default {default}]: ').strip()
    value = default if raw == '' else float(raw)
    if not 0.0 <= value <= 1.0:
        raise ValueError(f'{prompt} must be between 0 and 1.')
    return value

population_size = read_int('Population size', 100, 4)
generations = read_int('Number of generations', 300, 1)
crossover_rate = read_probability('Crossover rate', 0.9)
mutation_rate = read_probability('Mutation rate', 0.15)

print('\nGA parameters:')
print(f'Population = {population_size}, generations = {generations}')
print(f'Crossover rate = {crossover_rate}, mutation rate = {mutation_rate}')


GA parameters:
Population = 100, generations = 300
Crossover rate = 0.9, mutation rate = 0.15


## 4. Genetic Algorithm

This implementation uses **tournament selection**, **ordered crossover** (which keeps each city only once), **swap mutation**, and **elitism** (carry the two best routes into the next generation).

In [7]:
random.seed(42)
np.random.seed(42)

def tournament_selection(population, tournament_size=3):
    """Choose the shortest route among a random sample of the population."""
    contestants = random.sample(population, min(tournament_size, len(population)))
    return min(contestants, key=route_distance).copy()

def ordered_crossover(parent1, parent2):
    """Ordered crossover (OX), preserving a valid permutation of the cities."""
    size = len(parent1)
    left, right = sorted(random.sample(range(size), 2))
    child = [None] * size
    child[left:right + 1] = parent1[left:right + 1]

    remaining = [gene for gene in parent2 if gene not in child]
    fill_positions = list(range(right + 1, size)) + list(range(0, left))
    for position, gene in zip(fill_positions, remaining):
        child[position] = gene
    return child

def swap_mutation(route):
    """Swap two city positions in a route."""
    i, j = random.sample(range(len(route)), 2)
    route[i], route[j] = route[j], route[i]
    return route

# Initial population: random valid tours
population = [random.sample(range(n_cities), n_cities) for _ in range(population_size)]
best_route = min(population, key=route_distance).copy()
best_distance = route_distance(best_route)
best_distance_history = [best_distance]

elite_count = min(2, population_size - 1)

for generation in range(generations):
    population.sort(key=route_distance)  # shortest routes first
    next_population = [route.copy() for route in population[:elite_count]]

    while len(next_population) < population_size:
        parent1 = tournament_selection(population)
        parent2 = tournament_selection(population)

        if random.random() < crossover_rate:
            child = ordered_crossover(parent1, parent2)
        else:
            child = parent1.copy()

        if random.random() < mutation_rate:
            child = swap_mutation(child)

        next_population.append(child)

    population = next_population
    generation_best = min(population, key=route_distance)
    generation_best_distance = route_distance(generation_best)

    if generation_best_distance < best_distance:
        best_route = generation_best.copy()
        best_distance = generation_best_distance

    best_distance_history.append(best_distance)

print('Genetic Algorithm finished.')

Genetic Algorithm finished.


## 5. Best route found

This is the best route discovered by the GA. A genetic algorithm is a heuristic, so its result is not guaranteed to be the mathematically optimal tour.

In [8]:
closed_route = best_route + [best_route[0]]
route_labels = [city_names[i] for i in closed_route]

print('Best route:', ' -> '.join(route_labels))
print(f'Minimum total distance found: {best_distance:.3f}')
print(f'Best fitness (1 / distance): {fitness(best_route):.6f}')

Best route: A -> G -> D -> E -> H -> C -> B -> F -> A
Minimum total distance found: 25.319
Best fitness (1 / distance): 0.039496


## 6. Visualize the route

The plotted line shows the order in which the cities are visited, including the return to the starting city.

In [ ]:
plt.figure(figsize=(8, 6))
route_coordinates = coordinates[closed_route]
plt.plot(route_coordinates[:, 0], route_coordinates[:, 1], marker='o', linewidth=1.8)

for i, name in enumerate(city_names):
    plt.annotate(name, (coordinates[i, 0], coordinates[i, 1]), xytext=(5, 5), textcoords='offset points')

plt.title(f'Best TSP Tour Found by GA (distance = {best_distance:.3f})')
plt.xlabel('x coordinate')
plt.ylabel('y coordinate')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(len(best_distance_history)), best_distance_history)
plt.title('GA Convergence: Best Distance vs Generation')
plt.xlabel('Generation')
plt.ylabel('Best total distance so far')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()